<a href="https://colab.research.google.com/github/mcarden4-netizen/Homework-Intro-to-ML/blob/Homework/Homework7part2a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%matplotlib inline

from matplotlib import pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import time
import copy

In [2]:
torch.manual_seed(123)
np.random.seed(123)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Using device:", device)

Using device: cpu


In [3]:
data = './data'

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer','dog', 'frog', 'horse', 'ship', 'truck']

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.4915, 0.4823, 0.4468),(0.2470, 0.2435, 0.2616))])

cifar10 = datasets.CIFAR10(data, train=True, download=True, transform=transform)

cifar10_val = datasets.CIFAR10(data, train=False, download=True, transform=transform)

100%|██████████| 170M/170M [23:20<00:00, 122kB/s]


In [4]:
Ts = len(cifar10)
Vs = int(0.1 * Ts)

cifar10t, cifar10v = random_split(cifar10, [Ts - Vs, Vs], generator=torch.Generator().manual_seed(50))

batch_size = 64

train_loader = DataLoader(cifar10t, batch_size=batch_size, shuffle=True)

train_eval_loader = DataLoader(cifar10t, batch_size=batch_size, shuffle=False)

val_loader = DataLoader(cifar10v, batch_size=batch_size, shuffle=False)

test_loader = DataLoader(cifar10_val, batch_size=batch_size, shuffle=False)

In [6]:
model3 = nn.Sequential(
    nn.Linear(3072, 512),
    nn.ReLU(),

    nn.Linear(512, 256),
    nn.ReLU(),

    nn.Linear(256, 128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 10)
).to(device)


In [7]:
def evaluate(model, loader, loss_fn):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for imgs, labels in loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs.view(imgs.shape[0], -1))

            loss = loss_fn(outputs, labels)

            total_loss += loss.item() * imgs.shape[0]

            _, predicted = torch.max(outputs, dim=1)

            total += labels.shape[0]
            correct += (predicted == labels).sum().item()

    average_loss = total_loss / total
    accuracy = 100 * correct / total

    return average_loss, accuracy

In [11]:
def train_model(model, model_name):

    loss_fn = nn.CrossEntropyLoss()

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    n_epochs = 100
    patience = 50

    best_val_loss = float('inf')
    best_model = copy.deepcopy(model.state_dict())

    epochs_without_improvement = 0

    train_losses = []
    val_losses = []

    start_time = time.time()

    for epoch in range(n_epochs):

        model.train()

        total_loss = 0
        total = 0

        for imgs, labels in train_loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs.view(imgs.shape[0], -1))

            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * imgs.shape[0]
            total += labels.shape[0]

        train_loss = total_loss / total

        val_loss, val_accuracy = evaluate(model, val_loader, loss_fn)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_model = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        if (epoch + 1) % 5 == 0:

            print(
                f'{model_name} | Epoch {epoch + 1}/{n_epochs} '
                f'| Train Loss: {train_loss:.4f} '
                f'| Validation Loss: {val_loss:.4f} '
                f'| Validation Accuracy: {val_accuracy:.2f}%'
            )

        if epochs_without_improvement >= patience:

            print(f'{model_name}: Early stopping at epoch {epoch + 1}')
            break

    training_time = time.time() - start_time

    model.load_state_dict(best_model)

    final_train_loss, train_accuracy = evaluate(model, train_eval_loader, loss_fn)

    final_val_loss, val_accuracy = evaluate(model, val_loader, loss_fn)

    test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)

    parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("\n", model_name, "Final Results")
    print("--------------------------------")
    print(f"Best Epoch: {best_epoch}")
    print(f"Training Time: {training_time:.2f} seconds")
    print(f"Training Loss: {final_train_loss:.4f}")
    print(f"Training Accuracy: {train_accuracy:.2f}%")
    print(f"Validation Accuracy: {val_accuracy:.2f}%")
    print(f"Test Accuracy: {test_accuracy:.2f}%")
    print(f"Trainable Parameters: {parameters:,}")

    results = {
        'name': model_name,
        'training_time': training_time,
        'train_loss': final_train_loss,
        'train_accuracy': train_accuracy,
        'val_accuracy': val_accuracy,
        'test_accuracy': test_accuracy,
        'parameters': parameters,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_epoch': best_epoch
    }

    return results

In [12]:
results3 = train_model(model3, "Problem 2.a - Ten Hidden Layers")

Problem 2.a - Ten Hidden Layers | Epoch 5/100 | Train Loss: 0.9282 | Validation Loss: 1.4822 | Validation Accuracy: 51.64%
Problem 2.a - Ten Hidden Layers | Epoch 10/100 | Train Loss: 0.7256 | Validation Loss: 1.6565 | Validation Accuracy: 52.46%
Problem 2.a - Ten Hidden Layers | Epoch 15/100 | Train Loss: 0.5771 | Validation Loss: 1.8352 | Validation Accuracy: 52.88%
Problem 2.a - Ten Hidden Layers | Epoch 20/100 | Train Loss: 0.4736 | Validation Loss: 2.0109 | Validation Accuracy: 52.00%
Problem 2.a - Ten Hidden Layers | Epoch 25/100 | Train Loss: 0.3947 | Validation Loss: 2.1136 | Validation Accuracy: 52.66%
Problem 2.a - Ten Hidden Layers | Epoch 30/100 | Train Loss: 0.3337 | Validation Loss: 2.1784 | Validation Accuracy: 52.96%
Problem 2.a - Ten Hidden Layers | Epoch 35/100 | Train Loss: 0.2992 | Validation Loss: 2.3223 | Validation Accuracy: 52.90%
Problem 2.a - Ten Hidden Layers | Epoch 40/100 | Train Loss: 0.2550 | Validation Loss: 2.5501 | Validation Accuracy: 51.94%
Problem 2